In [1]:
from tqdm.notebook import trange
import time
import random
import numpy as np 
import random as rd
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
# Notice that NUMBER_ROWS = NUMBER_COLS

torch.manual_seed(0)
from numba import njit
import warnings
from numba.core.errors import NumbaDeprecationWarning, NumbaPendingDeprecationWarning, NumbaExperimentalFeatureWarning, NumbaWarning

warnings.simplefilter('ignore', category = NumbaDeprecationWarning)
warnings.simplefilter('ignore', category = NumbaPendingDeprecationWarning)
warnings.simplefilter('ignore', category = NumbaExperimentalFeatureWarning)
warnings.simplefilter('ignore', category = NumbaWarning)

In [2]:
NUMBER_ROWS = 15
NUMBER_COLS = 15
NUMBER_PLAYERS = 2
NUMBER_ACTIONS = NUMBER_ROWS * NUMBER_COLS
ENV_SIZE = NUMBER_ROWS * NUMBER_COLS + 3
STATE_SIZE = NUMBER_ROWS * NUMBER_COLS + 3

@njit()
def init_env():
    env_state = np.full(ENV_SIZE, 0)
    return env_state

@njit()
def get_state_size():
    return STATE_SIZE

@njit()
def get_action_size():
    return NUMBER_ACTIONS

@njit()
def get_agent_size():
    return NUMBER_PLAYERS

@njit()
def convert_to_1D(i, j):
    return i * NUMBER_COLS + j

@njit()
def convert_to_2D(act):
    x = int(act / NUMBER_COLS)
    y = act - x * NUMBER_COLS
    return x, y

@njit()
def get_opponent_player(player):
    return 1 if player == 0 else 0

@njit()
def get_opponent_value(value):
    return -value
@njit()
def get_agent_state( env_state):
    p_state = np.full(STATE_SIZE, 0)
    # Get board state
    p_state[0 : (NUMBER_ROWS * NUMBER_COLS)] = env_state[0 : (NUMBER_ROWS * NUMBER_COLS)]
    # Get last checked cell
    p_state[NUMBER_ROWS * NUMBER_COLS] = env_state[NUMBER_ROWS * NUMBER_COLS]
    p_state[NUMBER_ROWS * NUMBER_COLS + 1] = env_state[NUMBER_ROWS * NUMBER_COLS + 1]
    return p_state

@njit()
def get_valid_actions(player_state):
    # list_action = np.full(NUMBER_ACTIONS, 0)
    # list_action[np.where(player_state[0 : NUMBER_ROWS * NUMBER_COLS] == 0)] = 1
    # return list_action
    return (player_state[0 : NUMBER_ACTIONS] == 0).astype(np.uint8)

@njit()
def check_ended(env):

    # Case 1: check end
    # p_id = turn % 2
    x = env[NUMBER_ROWS * NUMBER_COLS]
    y = env[NUMBER_ROWS * NUMBER_COLS + 1]
    p_id = env[x * NUMBER_COLS + y] - 1
    # Check row
    count = 1
    d = 1
    while y + d < NUMBER_COLS and env[convert_to_1D(x, y + d)] == p_id + 1:
        count += 1
        if count == 5:
            return p_id
        d += 1
    d = 1
    while y - d > -1 and env[convert_to_1D(x, y - d)] == p_id + 1:
        count += 1
        if count == 5:
            return p_id
        d += 1
    # Check col
    count = 1
    d = 1
    while x + d < NUMBER_ROWS and env[convert_to_1D(x + d, y)] == p_id + 1:
        count += 1
        if count == 5:
            return p_id
        d += 1
    d = 1
    while x - d > -1 and env[convert_to_1D(x - d, y)] == p_id + 1:
        count += 1
        if count == 5:
            return p_id
        d += 1
    # Check diagonal line C1
    count = 1
    d = 1
    while x + d < NUMBER_ROWS and y + d < NUMBER_COLS and env[convert_to_1D(x + d, y + d)] == p_id + 1:
        count += 1
        if count == 5:
            return p_id
        d += 1
    d = 1
    while x - d > -1 and y - d > -1 and env[convert_to_1D(x - d, y - d)] == p_id + 1:
        count += 1
        if count == 5:
            return p_id
        d += 1
    # Check diagonal line C2
    count = 1
    d = 1
    while x + d < NUMBER_ROWS and y - d > -1 and env[convert_to_1D(x + d, y - d)] == p_id + 1:
        count += 1
        if count == 5:
            return p_id
        d += 1
    d = 1
    while x - d > -1 and y + d < NUMBER_COLS and env[convert_to_1D(x - d, y + d)] == p_id + 1:
        count += 1
        if count == 5:
            return p_id
        d += 1
    # Case 2: all tie
    if(env[NUMBER_ROWS * NUMBER_COLS + 2] == NUMBER_ROWS * NUMBER_COLS):
        return 2
    return -1

@njit()
def next_step(action, env_state):
    env = np.copy(env_state)
    if env[action] != 0:
        raise Exception('Action error!')
    else:
        x = env[NUMBER_ROWS * NUMBER_COLS]
        y = env[NUMBER_ROWS * NUMBER_COLS + 1]
        p_id = env[x * NUMBER_COLS + y] % 2
        env[action] = p_id + 1
        env[NUMBER_ROWS * NUMBER_COLS + 2] += 1

    x, y = convert_to_2D(action)
    env[NUMBER_ROWS * NUMBER_COLS] = x
    env[NUMBER_ROWS * NUMBER_COLS + 1] = y

    return env

@njit()
def numba_bot_random(p_state, per):
    arr_action = get_valid_actions(p_state)
    act_idx = np.random.choice(np.where(arr_action == 1)[0])
    return act_idx, per

@njit()
def numba_run_one_game(p_main, p_o, per, print_mode = False):
    env = init_env()
    _cc = 0
    while _cc < NUMBER_COLS * NUMBER_ROWS:
        p_idx = env[NUMBER_COLS * NUMBER_ROWS + 2] % 2
        p_state = get_agent_state(env)
        turn = env[NUMBER_COLS * NUMBER_ROWS + 2]
        if (print_mode):
            print('----------------------------------------------------------------------------------')
            if (turn % 2 == 0):
                print('Turn of player: X')
            elif (turn % 2 == 1):
                print('Turn of player: O')
        if (p_idx == 0):
            action, per = p_main(p_state, per)
        elif (p_idx == 1):
            action, per = p_o(p_state, per)
        env = next_step(action, env)
        if (print_mode):
            print('Checked cell: (', env[NUMBER_ROWS * NUMBER_COLS], ',', env[NUMBER_ROWS * NUMBER_COLS + 1], ')')
        _cc += 1
        if (check_ended(env) != -1):
            break

    winner = check_ended(env)
    if (print_mode):
        if winner == 2:
            print('\n---------------------- All tie! ----------------------')
        elif winner == 0:
            print('\n---------------------- Winner: X ----------------------')
        elif winner == 1:
            print('\n---------------------- Winner: O ----------------------')

    if (winner == 2):
        winner = -1
    return winner, per

@njit()
def numba_run_n_game(p0, p1, per, num_game, print_mode = False):
    win = [0, 0]
    for _n in range(num_game):
        first = rd.randint(0, 1)
        if (first == 0):
            winner, per = numba_run_one_game(p0, p1, per, print_mode)
        else:
            winner, per = numba_run_one_game(p1, p0, per, print_mode)
        if winner != -1:
            if (winner == 0):
                win[0] += 1 * (1 - first)
                win[1] += 1 * first
            elif (winner == 1):
                win[0] += 1 * first
                win[1] += 1 * (1 - first)
    if (print_mode):
        print()
    return win, per

In [3]:
@njit()
def get_encode_state(env):
    env = np.reshape(env[ : 225], (15, 15))
    encode_state = np.stack(
        (env == 2, env == 0, env == 1)
    ).astype(np.float32)
    return encode_state

@njit()
def change_perspective(env, player):
    n_env = np.copy(env)
    if player == 0: return n_env
    temp = np.where(n_env[0 : NUMBER_ROWS * NUMBER_COLS] == 2)
    n_env[np.where(n_env[0 : NUMBER_ROWS * NUMBER_COLS] == 1)] = 2
    n_env[temp] = 1
    return n_env

In [5]:
start = time.time()
win, per = numba_run_n_game(numba_bot_random, numba_bot_random, None, 1, False)
end = time.time()
print(win)
print(end - start)

[0, 1]
0.0009725093841552734


In [6]:
class Node:
    def __init__(self, args, env, parent = None, action_taken = -1, prior = 0.0, visit_count = 0):
        self.args = args
        self.env = env
        self.parent = parent
        self.action_taken = action_taken
        self.prior = prior
        self.children = []
        self.visit_count = visit_count
        self.value_sum = 0.0

    def is_fully_expanded(self):
        return len(self.children) > 0

    def select(self):
        best_child = None
        best_ucb = -np.inf

        for child in self.children:
            ucb = self.get_ucb(child)
            if ucb > best_ucb:
                best_ucb = ucb
                best_child = child
        return best_child

    def get_ucb(self, child):
        if child.visit_count == 0:
            q_value = 0
        else:
            q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
        return q_value + self.args['C'] * (math.sqrt(self.visit_count) / (child.visit_count + 1)) * child.prior

    def expand(self, policy):
        for act, prob in enumerate(policy):
            if prob > 0:
              child_env = next_step(act, self.env)
              child_env = change_perspective(child_env, 1)

              child = Node(self.args, child_env, self, act, prob)
              self.children.append(child)

    def backpropagate(self, value):
        self.value_sum += value
        self.visit_count += 1
        if self.parent is not None:
            self.parent.backpropagate(-value)

In [7]:
class MCTS:
    def __init__(self, args, model) -> None:
        # args: chứa những thông tin để sử dụng hoặc kết thúc thuật toán
        self.args = args
        self.model = model

    # Không dùng data để train ngay lập tức mà chỉ sử dụng resnet để dự đoán policy và value
    @torch.no_grad()
    def search(self, env):
        root = Node(self.args, env, visit_count = 1)

        # Thêm noise, tăng khả năng khai phá
        policy, _ = self.model(
            torch.tensor(get_encode_state(env), device = self.model.device).unsqueeze(0)
        )
        policy = torch.softmax(policy, axis = 1).squeeze(0).cpu().numpy()

        policy = (1 - self.args['dirichlet_esp']) * policy + self.args['dirichlet_esp'] * np.random.dirichlet([self.args['dirichlet_alp']] * NUMBER_ACTIONS)

        valid_acts = get_valid_actions(env)
        policy *= valid_acts
        policy /= np.sum(policy)
        root.expand(policy)
        for _search in range (self.args['num_searches']):
            # Selection phase:
            node = root
            while node.is_fully_expanded():
                node = node.select()

            # Prepare for next step
            check_win = check_ended(node.env) if not node.action_taken == -1 else -1
            ## Value ở đây luôn mang giá trị -1, vì người chơi hiện tại chưa đánh mà bàn cờ đã kết thúc => thua
            value = 0
            if check_win == 0 or check_win == 1:
                value = -1
            if check_win == -1:
                # Dùng mạng đưa ra policy và value
                policy, value = self.model(
                    torch.tensor(get_encode_state(node.env), device=self.model.device).unsqueeze(0)
                )
                policy = torch.softmax(policy, 1).squeeze(0).cpu().numpy()

                valid_move = get_valid_actions(node.env)
                policy *= valid_move
                policy /= np.sum(policy)
                value = value.item()
                # Expansion phase:
                node.expand(policy)
            # Backpropagation phase
            node.backpropagate(value)

        # Return the prior probalities
        action_probs = np.zeros(get_action_size())
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        del root
        return action_probs

In [8]:
class ResNet(nn.Module):
    def __init__(self, num_resBlock, num_hidden, device) -> None:
        super().__init__()
        self.device = device
        # Layer Input
        ## Chuyển hóa encode env thành các input cho layer backBone
        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        # Layer backBone bao gồm các RestBlock
        self.backBone = nn.ModuleList(
            [RestBlock(num_hidden) for _n in range(num_resBlock)]
        )

        ## Huấn luyện động thời 2 layer
        # Layer output policyHead đưa ra prior probability
        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * NUMBER_ROWS * NUMBER_COLS, NUMBER_ACTIONS)
        )

        # Layer output đưa ra giá trị của env
        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * NUMBER_ROWS * NUMBER_COLS, 1),
            nn.Tanh()
        )

        self.to(device)

    # Hàm mô quá đường đi của dữ encode env x khi xử lý qua mạng
    def forward(self, x):
        x = self.startBlock(x)
        for resBlock in self.backBone:
            x = resBlock.forward(x)
        policy = self.policyHead(x)
        value = self.valueHead(x)
        return policy, value

# Các block liên tiếp trong layer backBone của cấu trúc Residual neural network
class RestBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.batchn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.batchn2 = nn.BatchNorm2d(num_hidden)

    # _x được lấy dư và cộng đồng thời với output thông qua xử lý của RestBlock
    def forward(self, x):
        residual = x
        x = F.relu(self.batchn1(self.conv1(x)))
        x = self.batchn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x

In [9]:
def print_env(env):
    for i in range(NUMBER_COLS):
        print(env[i * 15 : i * 15 + 15])
    print(env[NUMBER_ROWS * NUMBER_COLS : NUMBER_ROWS * NUMBER_COLS + 3])

In [10]:
args = {
    'C': 5,
    'num_searches' : 150,
    'num_iterations' : 1,
    'num_selfPlay_iterations' : 1,
    'num_epochs' : 0,
    'batch_size' : 64,

    'temperature' : 1.25,
    'dirichlet_esp' : 0.25,
    'dirichlet_alp' : 0.35
}
def one_game_pvc():
    model = ResNet(4, 64, device = torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()
    mcts = MCTS(args, model)
    env = init_env()
    while True:
        if env[NUMBER_ROWS * NUMBER_COLS + 2] % 2 == 0:
            valid_moves = get_valid_actions(env)
            act = int(input("Choose action: "))
            if valid_moves[act] == 0:
                print("action not valid")
                continue
        else:
            neutral_state = change_perspective(env, 1)
            mcts_probs = mcts.search(neutral_state)
            act = np.argmax(mcts_probs)

        env = next_step(act, env)
        check_end = check_ended(env)
        if check_end != -1:
            if check_end == 2:
                print('\n---------------------- All tie! ----------------------')
            elif check_end == 0:
                print('\n---------------------- Winner: Human ----------------------')
            elif check_end == 1:
                print('\n---------------------- Winner: Comp ----------------------')

            break
one_game_pvc()


---------------------- Winner: Human ----------------------


In [15]:
class AlphaGomoku:
    def __init__(self, model, optimizer, args):
        self.model = model
        self.optimizer = optimizer
        self.args = args
        self.mcts = MCTS(args, model)

    def selfPlay(self):
        memory = []
        env = init_env()
        player = 0
        while True:
              neutral_env = change_perspective(env, player)
              action_prob = self.mcts.search(neutral_env)
              memory.append((neutral_env, action_prob, player))

              # temp_probs nhằm tăng hoặc giảm khoảng cách xác suất giữa các nước đi
              ## Điều chỉnh để có thể ưu tiên tìm kiếm hoặc ưu tiên khai thác
              temperature_action_probs = get_temp_act_probs(action_prob, self.args['temperature'])

              action = np.random.choice(NUMBER_ACTIONS, p=temperature_action_probs)
              env = next_step(action, env)
              value = check_ended(env)
              if value != -1:
                  value = 0 if value == 2 else 1
                  returnMemory = []
                  for his_env, his_act_prob, his_player in memory:
                      his_outcome = value if his_player == player else get_opponent_value(value)
                      returnMemory.append((
                          get_encode_state(his_env),
                          his_act_prob,
                          his_outcome
                      ))
                  return returnMemory
              player = get_opponent_player(player)

    def train(self, memory):
        random.shuffle(memory)
        for batch_ind in range(0, len(memory), self.args['batch_size']):
            sample = memory[batch_ind : min(len(memory) - 1, batch_ind + self.args['batch_size'])]
            states, policy_targets, value_targets = zip(*sample)

            states, policy_targets, value_targets = np.array(states), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)
            states = torch.tensor(states, dtype = torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype = torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype = torch.float32, device=self.model.device)

            out_policy, out_value = self.model(states)

            # Đánh giá sự mất mát của hàm cross_entropy ( hàm đo lường mức độ tương tự giữa phân phối xác suất của mạng đưa ra và phân phối thực tế )
            policy_loss = F.cross_entropy(out_policy, policy_targets)

            # Đánh giá sự mất mát theo phương thức bình phương sai số trung bình của giá trị bàn cờ giữa giá trị thực và giá trị của mạng đưa ra
            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss

            # Điều chỉnh gradient về 0
            self.optimizer.zero_grad()
            # Tính toán gradient và lan truyền ngược
            loss.backward()
            # Cập nhật trọng số của mô hình
            self.optimizer.step()

    def play_with_bot(self, another_bot):
        memory = []
        alpha_player = rd.randint(0, 1)
        env = init_env()
        while True:
            if env[NUMBER_ACTIONS + 2] % 2 == alpha_player:
                neutral_env = change_perspective(env, alpha_player)
                action_prob = self.mcts.search(neutral_env)
                memory.append((neutral_env, action_prob))
                temperature_action_probs = get_temp_act_probs(action_prob, self.args['temperature'])

                action = np.random.choice(NUMBER_ACTIONS, p=temperature_action_probs)
            else:
                # Thay đổi hàm lấy action của bot tương ứng ở đây
                action, _ = another_bot(env, None)
            env = next_step(action, env)
            value = check_ended(env)
            if value != -1:
                returnMemory = []
                if value == 2:
                    his_outcome = 0
                else:
                    his_outcome = 1 if value == alpha_player else -1
                for his_env, his_act_prob in memory:
                    returnMemory.append((
                        get_encode_state(his_env),
                        his_act_prob,
                        his_outcome
                    ))
                return returnMemory

    def learn(self, another_bot = None):
        for i in range(self.args['num_iterations']):
            memory = []
            self.model.eval()
            if another_bot is None:
                for selfPlay_i in trange(self.args['num_selfPlay_iterations']):
                    memory += self.selfPlay() 
            else:
                for selfPlay_i in trange(self.args['num_selfPlay_iterations']):
                    memory += self.play_with_bot(another_bot) 
            print(len(memory))
            self.model.train()
            for epoch in trange(self.args['num_epochs']):
                self.train(memory)

            torch.save(self.model.state_dict(), f"model_{i}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{i}.pt")

    def log_win_rate(self, another_bot, num_game = 1000):
        self.model.eval()
        num_game_alpha_win = 0
        num_game_alpha_draw = 0
        for game in trange(num_game):
            env = init_env()
            alpha_player = rd.randint(0, 1)
            while True:
                if env[NUMBER_ACTIONS + 2] % 2 == alpha_player:
                    neutral_env = change_perspective(env, alpha_player)

                    # Có thể sử dụng trực tiếp luôn policy lấy từ mạng
                    action_prob, _ = self.model(torch.tensor(get_encode_state(neutral_env), device = self.model.device).unsqueeze(0))
                    action_prob = torch.softmax(action_prob, axis = 1).squeeze(0).detach().cpu().numpy()
                    
                    valid_move = get_valid_actions(env)
                    action_prob *= valid_move
                    action_prob /= np.sum(action_prob)

                    # Hoặc sử dụng tìm kiếm của MCTS ( lâu hơn )
                    # action_prob = self.mcts.search(neutral_env)

                    action = np.argmax(action_prob)
                else:
                    # Thay đổi hàm lấy action của bot tương ứng ở đây
                    action, _ = another_bot(env, None)
                env = next_step(action, env)
                value = check_ended(env)
                if value != -1:
                    if value == alpha_player:
                        num_game_alpha_win += 1
                    elif value == 2:
                        num_game_alpha_draw += 1
                    break

        print("AlphaGomoku win: {}/{} game, draw: {}/{} game!".format(num_game_alpha_win, num_game, num_game_alpha_draw, num_game))
@njit()
def get_temp_act_probs(action_prob, t):
    action_prob = action_prob ** (1 / t)
    action_prob /= np.sum(action_prob)
    return action_prob

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet(4, 64, device)
# Có thể sử dụng load_state_dict để lấy model cũ tiếp tục huấn luyện
model.load_state_dict(torch.load('model_0.pt', map_location=device))
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001, weight_decay = 0.0001)

# Điều chỉnh các tham số phù hợp
args = {
    'C': 6,
    'num_searches' : 300,
    'num_iterations' : 1,
    'num_selfPlay_iterations' : 200,
    'num_epochs' : 11,
    'batch_size' : 64,

    'temperature' : 1.25,
    'dirichlet_esp' : 0.25,
    'dirichlet_alp' : 0.35
}
alphaGomoku = AlphaGomoku(model, optimizer, args)
alphaGomoku.log_win_rate(numba_bot_random)

  0%|          | 0/1000 [00:00<?, ?it/s]

AlphaGomoku win: 654/1000 game, draw: 0/1000 game!


In [27]:
args = {
    'C': 5,
    'num_searches' : 150,
    'num_iterations' : 1,
    'num_selfPlay_iterations' : 100,
    'num_epochs' : 20,
    'batch_size' : 64,

    'temperature' : 1.25,
    'dirichlet_esp' : 0.25,
    'dirichlet_alp' : 0.35
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet(4, 64, device)
model.load_state_dict(torch.load('model_0.pt', map_location=device))
model.eval()

env = init_env()
policy, value = model(torch.tensor(get_encode_state(env), device=device).unsqueeze(0))
value = value.item()
policy = torch.softmax(policy, 1).squeeze(0).detach().cpu().numpy()
va = get_valid_actions(env)
policy *= va
print(value)
print(policy)
print(np.argmax(policy))

0.6863303184509277
[2.98003317e-04 4.17344825e-04 4.25014674e-04 1.28757936e-04
 1.07624440e-03 2.17346693e-04 5.12786093e-04 1.52835302e-04
 1.55510774e-04 3.08349176e-04 4.31099994e-04 8.22566799e-04
 2.15861655e-04 7.20472366e-04 1.50011620e-04 2.23231851e-04
 2.95028643e-04 5.85714995e-04 5.12363738e-04 6.07688504e-04
 1.69455830e-03 5.21001872e-04 5.29968122e-04 1.35826459e-03
 2.61816080e-04 1.92363013e-03 9.53297305e-04 2.89523188e-04
 2.88479868e-03 2.09594742e-04 4.39552619e-04 3.02097207e-04
 1.75242918e-03 4.09298780e-04 1.82954376e-04 2.49323988e-04
 3.04376561e-04 3.45337467e-04 1.39511190e-04 5.97801176e-04
 1.32114190e-04 5.47897653e-04 1.68634608e-04 1.48270861e-04
 5.91933203e-04 1.05987441e-04 2.54237209e-04 1.84139673e-04
 2.62017536e-04 7.94637192e-04 1.87731313e-03 2.57875770e-04
 2.15453721e-04 2.94213532e-04 1.50224369e-04 5.38123655e-04
 9.16393846e-03 7.00718374e-04 1.38662261e-04 1.39698992e-03
 4.30487038e-04 5.62496192e-04 4.02441656e-04 1.25749357e-04
 1.46

1.6371097564697266
